# 03 · Build the golden set
Sample 200 held-out messages, hand-label into the 7 intents, flag ambiguous/mixed cases.

Labels + protocol live in `src/build_golden.py` (index-aligned to a seed=7 sample of the held-out pool). This notebook inspects the labeled set.

In [1]:
import os, sys
os.environ['PYTHONUTF8'] = '1'
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')
import pandas as pd
from src import build_golden
if not os.path.exists('data/golden_set.csv'):
    build_golden.main()
gold = pd.read_csv('data/golden_set.csv')
print(len(gold), 'labeled examples')
gold.head()

200 labeled examples


,pair_id,customer_msg,true_intent,ambiguous,notes
0,uc_1097878,@Uber_Support I have reached out through email...,general_query,True,mixed/context-dependent
1,uc_1794401,Craving dumplings. Help a sister out @115877 👋🏼,delivery_order,False,NaN
2,uc_2907302,I am pissed with @115873 !! My driver was 3 mi...,trip_issue,False,NaN
3,uc_393638,@Uber_Support what do I do when I get charged ...,billing_payment,False,NaN
4,uc_1449727,@115873 why are the airports trying to ruin y'...,trip_issue,False,NaN


## Label distribution — intentionally imbalanced (mirrors real Uber traffic)

In [2]:
print(gold['true_intent'].value_counts())
print(f"\nambiguous/mixed: {gold['ambiguous'].sum()} ({gold['ambiguous'].mean()*100:.0f}%)")

true_intent
billing_payment      55
general_query        41
trip_issue           31
service_complaint    31
delivery_order       16
account_access       15
safety_incident      11
Name: count, dtype: int64

ambiguous/mixed: 44 (22%)


## Labeling notes
- Sampled from the **held-out pool** (disjoint from retrieval corpus) → no leakage into eval.
- Mixed billing/trip rule: concrete ask = refund → `billing_payment`; ride/driver experience → `trip_issue`.
- UberEats: food quality/missing → `delivery_order`; refund ask → `billing_payment`.
- Ambiguous items kept but flagged, so we can report full-set vs non-ambiguous metrics separately.

In [3]:
# a few ambiguous examples
for _, r in gold[gold['ambiguous']].head(6).iterrows():
    print(f"[{r['true_intent']}] {r['customer_msg'][:120]}")

[general_query] @Uber_Support I have reached out through email and the app
[general_query] @Uber_Support But I need to know now this is a scheduled ride for Friday.
[billing_payment] @Uber_Support There’s no place to write a msg. One: driver didn’t alert me he was here and left. and charged me a cancel
[general_query] @Uber_Support I follow up at that link. Looking forward to reply. I hope you don’t penalize drivers for following the ba
[billing_payment] @Uber_Support I had a question about my billing for UberEats, added a gift card into my account but you guys charged my 
[service_complaint] So @115873 and @Uber_Support decide to cheat me off my money and then disgust me with these responses. #UberCheat #UberI
